In [4]:
# ============================================
# الخلية 1: تثبيت المكتبات وتجهيز البيئة
# ============================================

import os
import torch
import yaml
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import zipfile
import patoolib
import shutil
from tqdm import tqdm
import random
from datetime import datetime
import subprocess
import sys

print("="*60)
print("🔧 تثبيت المكتبات وتجهيز البيئة")
print("="*60)

# تثبيت المكتبات
!apt-get update -qq
!apt-get install -y unrar p7zip-full -qq
!pip install -q ultralytics patool opencv-python wget

print("\n✅ تم تثبيت المكتبات")

# التحقق من GPU
print(f"\n🎮 GPU متاح: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   • GPU: {torch.cuda.get_device_name(0)}")
    print(f"   • الذاكرة: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

🔧 تثبيت المكتبات وتجهيز البيئة
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 4, in <module>
    from pip._internal.cli.main import main
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/build_env.py", line 19, in <module>
    from pip._internal.cli.spinners import open_spinner
  File

In [ ]:
# ============================================
# الخلية 2: فك ضغط Self-Driving Cars Dataset
# ============================================

print("="*60)
print("🚗 فك ضغط Self-Driving Cars Dataset")
print("="*60)

# مسار الملف المضغوط
car_rar_path = '/content/car.rar'
car_extract_path = '/content/car_dataset'

# التحقق من وجود الملف
if not os.path.exists(car_rar_path):
    print(f"\n❌ الملف غير موجود: {car_rar_path}")
    print("🔍 البحث عن الملف...")
    !find /content -name "*.rar" -type f
    found = !find /content -name "*.rar" -type f | head -1
    if found:
        car_rar_path = found[0]
        print(f"✅ تم العثور على: {car_rar_path}")
    else:
        raise Exception("لم يتم العثور على ملف car.rar")

print(f"\n📦 ملف RAR: {car_rar_path}")
print(f"📂 مجلد الاستخراج: {car_extract_path}")

# إنشاء مجلد الاستخراج
os.makedirs(car_extract_path, exist_ok=True)

# فك الضغط
try:
    print("\n⏳ جاري فك الضغط...")
    patoolib.extract_archive(car_rar_path, outdir=car_extract_path)
    print("✅ تم فك الضغط بنجاح!")
except Exception as e:
    print(f"⚠️ فشل patool: {e}")
    !unrar x {car_rar_path} {car_extract_path}/ -y
    print("✅ تم فك الضغط باستخدام unrar!")

In [5]:
# ============================================
# الخلية 3: تحميل GTSDB
# ============================================

print("="*60)
print("🇩🇪 تحميل GTSDB (German Traffic Sign Detection Benchmark)")
print("="*60)

gtsdb_zip_path = '/content/GTSDB.zip'
gtsdb_extract_path = '/content/gtsdb_dataset'

if not os.path.exists(gtsdb_zip_path):
    print("\n⏳ جاري تحميل GTSDB (1.5 GB)...")
    url = "https://benchmark.ini.rub.de/gtsdb_dataset/GTSDB.zip"
    wget.download(url, gtsdb_zip_path)
    print("\n✅ تم التحميل!")
else:
    print(f"\n✅ GTSDB.zip موجود بالفعل")

# فك ضغط GTSDB
print("\n⏳ جاري فك ضغط GTSDB...")
os.makedirs(gtsdb_extract_path, exist_ok=True)
with zipfile.ZipFile(gtsdb_zip_path, 'r') as zip_ref:
    zip_ref.extractall(gtsdb_extract_path)
print("✅ تم فك الضغط!")

# البحث عن gt.txt
gt_file = None
gtsdb_img_dir = None
for root, dirs, files in os.walk(gtsdb_extract_path):
    if 'gt.txt' in files:
        gt_file = os.path.join(root, 'gt.txt')
        gtsdb_img_dir = root
        break

if gt_file:
    print(f"\n✅ تم العثور على gt.txt في: {gt_file}")
else:
    raise Exception("لم يتم العثور على gt.txt في GTSDB")

🇩🇪 تحميل GTSDB (German Traffic Sign Detection Benchmark)

⏳ جاري تحميل GTSDB (1.5 GB)...


HTTPError: HTTP Error 404: Not Found

In [ ]:
# ============================================
# الخلية 4: تحليل بنية Self-Driving Cars Dataset
# ============================================

def analyze_car_structure():
    """تحليل بنية مجلدات Self-Driving Cars"""

    print("="*60)
    print("🔍 تحليل بنية Self-Driving Cars Dataset")
    print("="*60)

    splits = {}

    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(car_extract_path, split, 'images')
        label_dir = os.path.join(car_extract_path, split, 'labels')

        if os.path.exists(img_dir):
            images = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
            labels = os.listdir(label_dir) if os.path.exists(label_dir) else []

            splits[split] = {
                'images': img_dir,
                'labels': label_dir if os.path.exists(label_dir) else None,
                'num_images': len(images),
                'num_labels': len(labels)
            }

            print(f"\n📂 {split.upper()}:")
            print(f"   • الصور: {len(images)}")
            print(f"   • العلامات: {len(labels)}")
            print(f"   • مجلد الصور: {img_dir}")

    return splits

car_splits = analyze_car_structure()

In [ ]:
# ============================================
# الخلية 5: قراءة علامات GTSDB
# ============================================

def read_gtsdb_annotations(gt_file):
    """قراءة علامات GTSDB من ملف gt.txt"""
    annotations = {}

    print("="*60)
    print("📖 قراءة علامات GTSDB")
    print("="*60)

    with open(gt_file, 'r') as f:
        for line in tqdm(f, desc="معالجة GTSDB"):
            parts = line.strip().split(';')
            if len(parts) >= 6:
                img_name = parts[0]
                xmin = int(parts[1])
                ymin = int(parts[2])
                xmax = int(parts[3])
                ymax = int(parts[4])
                class_id = int(parts[5])

                if img_name not in annotations:
                    annotations[img_name] = []

                annotations[img_name].append({
                    'bbox': [xmin, ymin, xmax-xmin, ymax-ymin],
                    'class_id': class_id  # 0-42
                })

    print(f"\n📸 صور GTSDB: {len(annotations)}")
    total_signs = sum(len(v) for v in annotations.values())
    print(f"🚦 إجمالي الإشارات: {total_signs}")

    return annotations

gtsdb_annotations = read_gtsdb_annotations(gt_file)

In [ ]:
# ============================================
# الخلية 6: دمج الفئات
# ============================================

# فئات GTSDB (43 فئة)
gtsdb_classes = [f'GTSDB_Class_{i:02d}' for i in range(43)]

# فئات Self-Driving Cars (15 فئة)
car_classes = [
    'Green Light', 'Red Light', 'Speed Limit 10', 'Speed Limit 100',
    'Speed Limit 110', 'Speed Limit 120', 'Speed Limit 20', 'Speed Limit 30',
    'Speed Limit 40', 'Speed Limit 50', 'Speed Limit 60', 'Speed Limit 70',
    'Speed Limit 80', 'Speed Limit 90', 'Stop'
]

# دمج الفئات
all_classes = gtsdb_classes + car_classes
num_classes = len(all_classes)

print("="*60)
print("📝 دمج الفئات")
print("="*60)
print(f"\n📊 إجمالي الفئات: {num_classes}")
print(f"   • GTSDB: 43 فئة (0-42)")
print(f"   • Self-Driving Cars: 15 فئة (43-57)")

# إنشاء mapping
class_mapping = {i: name for i, name in enumerate(all_classes)}
print(f"\n📋 أول 10 فئات:")
for i in range(10):
    print(f"   • {i}: {class_mapping[i]}")

In [ ]:
# ============================================
# الخلية 7: إنشاء مجلد YOLO موحد
# ============================================

def convert_to_yolo_format(x, y, width, height, img_width, img_height):
    """تحويل إحداثيات لصيغة YOLO"""
    x_center = (x + width/2) / img_width
    y_center = (y + height/2) / img_height
    w_norm = width / img_width
    h_norm = height / img_height

    x_center = max(0, min(1, x_center))
    y_center = max(0, min(1, y_center))
    w_norm = max(0, min(1, w_norm))
    h_norm = max(0, min(1, h_norm))

    return x_center, y_center, w_norm, h_norm

# إنشاء مجلد موحد
merged_dir = '/content/merged_dataset'
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(merged_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(merged_dir, 'labels', split), exist_ok=True)

print("="*60)
print("🔄 إنشاء مجلد YOLO موحد")
print("="*60)
print(f"📂 المسار: {merged_dir}")

# دمج GTSDB (كلها في train)
print("\n⏳ دمج GTSDB...")
gtsdb_added = 0
for img_name, annotations in tqdm(gtsdb_annotations.items(), desc="GTSDB"):
    img_path = os.path.join(gtsdb_img_dir, img_name)
    img = cv2.imread(img_path)
    if img is None:
        continue

    h, w = img.shape[:2]

    # حفظ الصورة
    new_img_name = f"gtsdb_{img_name.replace('.ppm', '.jpg')}"
    dst_img = os.path.join(merged_dir, 'images', 'train', new_img_name)
    cv2.imwrite(dst_img, img)

    # حفظ العلامات
    label_file = os.path.join(merged_dir, 'labels', 'train', new_img_name.replace('.jpg', '.txt'))

    with open(label_file, 'w') as f:
        for ann in annotations:
            x, y, bw, bh = ann['bbox']
            class_id = ann['class_id']  # 0-42

            xc, yc, wn, hn = convert_to_yolo_format(x, y, bw, bh, w, h)
            f.write(f"{class_id} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

    gtsdb_added += 1

print(f"✅ تمت إضافة {gtsdb_added} صورة GTSDB")

# دمج Self-Driving Cars
print("\n⏳ دمج Self-Driving Cars...")
car_added = 0
car_skipped = 0

for split_name, split_info in car_splits.items():
    img_dir = split_info['images']
    label_dir = split_info['labels']

    # تحديد المجلد الوجهة
    if split_name == 'train':
        dest_split = 'train'
    elif split_name == 'valid':
        dest_split = 'val'
    else:
        dest_split = 'test'

    if not os.path.exists(img_dir):
        continue

    images = os.listdir(img_dir)

    for img_name in tqdm(images, desc=f"Self-Driving {split_name}"):
        img_path = os.path.join(img_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            car_skipped += 1
            continue

        h, w = img.shape[:2]

        # البحث عن ملف العلامات
        label_name = img_name.replace('.jpg', '.txt').replace('.png', '.txt')
        label_path = os.path.join(label_dir, label_name) if label_dir else None

        if not label_path or not os.path.exists(label_path):
            car_skipped += 1
            continue

        # حفظ الصورة
        new_img_name = f"car_{img_name}"
        dst_img = os.path.join(merged_dir, 'images', dest_split, new_img_name)
        cv2.imwrite(dst_img, img)

        # حفظ العلامات (مع إزاحة class_id +43)
        label_file = os.path.join(merged_dir, 'labels', dest_split, new_img_name.replace('.jpg', '.txt'))

        with open(label_file, 'w') as f:
            with open(label_path, 'r') as label_f:
                for line in label_f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        class_id = int(parts[0]) + 43  # إزاحة
                        xc, yc, wn, hn = map(float, parts[1:5])
                        f.write(f"{class_id} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

        car_added += 1

print(f"✅ تمت إضافة {car_added} صورة Self-Driving Cars")
print(f"⚠️ تم تخطي {car_skipped} صورة")

In [ ]:
# ============================================
# الخلية 8: إنشاء ملف data.yaml
# ============================================

data_config = {
    'path': merged_dir,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': num_classes,
    'names': all_classes
}

yaml_path = '/content/merged_dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("="*60)
print("📝 ملف data.yaml للمجموعة المدمجة")
print("="*60)
print(f"\n📁 المسار: {yaml_path}")
print(f"\n📋 المحتوى:")
print("-"*40)
!cat {yaml_path}

In [ ]:
# ============================================
# الخلية 9: إحصائيات المجموعة المدمجة
# ============================================

print("="*60)
print("📊 إحصائيات المجموعة المدمجة")
print("="*60)

total_train = len(os.listdir(os.path.join(merged_dir, 'images', 'train'))) if os.path.exists(os.path.join(merged_dir, 'images', 'train')) else 0
total_val = len(os.listdir(os.path.join(merged_dir, 'images', 'val'))) if os.path.exists(os.path.join(merged_dir, 'images', 'val')) else 0
total_test = len(os.listdir(os.path.join(merged_dir, 'images', 'test'))) if os.path.exists(os.path.join(merged_dir, 'images', 'test')) else 0

print(f"\n📂 TRAIN:")
print(f"   • الصور: {total_train}")

print(f"\n📂 VAL:")
print(f"   • الصور: {total_val}")

print(f"\n📂 TEST:")
print(f"   • الصور: {total_test}")

print(f"\n✅ الإجمالي: {total_train + total_val + total_test} صورة")
print(f"📊 عدد الفئات: {num_classes}")

In [ ]:
# ============================================
# الخلية 10: تدريب YOLO على المجموعة المدمجة
# ============================================

print("="*60)
print("🚀 بدء تدريب YOLOv8 على المجموعة المدمجة")
print("="*60)

# معلومات GPU
if torch.cuda.is_available():
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"   • الذاكرة: {total_memory:.1f} GB")

    # اختيار النموذج حسب الذاكرة
    if total_memory > 20:
        model = YOLO('yolov8x.pt')
        batch_size = 32
        print(f"✅ استخدام YOLOv8x")
    else:
        model = YOLO('yolov8m.pt')
        batch_size = 16
        print(f"✅ استخدام YOLOv8m")
else:
    model = YOLO('yolov8n.pt')
    batch_size = 8
    print(f"⚠️ استخدام YOLOv8n على CPU")

print(f"\n⚙️ إعدادات التدريب:")
print(f"   • حجم الدفعة: {batch_size}")
print(f"   • حجم الصورة: 640")
print(f"   • عدد الفئات: {num_classes}")

# بدء التدريب
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=batch_size,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=4,
    cache=True,
    patience=15,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.001,
    mosaic=1.0,
    mixup=0.2,
    fliplr=0.5,
    close_mosaic=10,
    optimizer='AdamW',
    lr0=0.0005,
    weight_decay=0.0005,
    warmup_epochs=3,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    project='/content/merged_training',
    name='yolov8_merged',
    exist_ok=True,
    verbose=True,
    seed=42
)

print("\n✅ التدريب قيد التشغيل!")